In [ ]:
!pip install transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

from datasets import Dataset

from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/balanced_reviews_6k.csv')

In [ ]:
df = df[['cleaned_review', 'sentiment']]

In [ ]:
df.head()

,cleaned_review,sentiment
0,instead of going to a jewelry store to purchas...,positive
1,sadly had to return as toe area too shallow we...,neutral
2,my husband and i both desperately needed slipp...,positive
3,well this dress was only about and you can tel...,neutral
4,this watch feels like cheap plastici gave it t...,negative


In [ ]:
label_map = {
    'negative': 0,
    'neutral': 1,
    'positive': 2
}

df['label'] = df['sentiment'].map(label_map)

In [ ]:
df.head()

,cleaned_review,sentiment,label
0,instead of going to a jewelry store to purchas...,positive,2
1,sadly had to return as toe area too shallow we...,neutral,1
2,my husband and i both desperately needed slipp...,positive,2
3,well this dress was only about and you can tel...,neutral,1
4,this watch feels like cheap plastici gave it t...,negative,0


In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['cleaned_review'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained(
    'distilbert-base-uncased'
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True
)

In [ ]:
train_dataset = Dataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': list(train_labels)
})

test_dataset = Dataset.from_dict({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask'],
    'labels': list(test_labels)
})

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=3
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average='weighted'
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',

    eval_strategy='epoch',

    save_strategy='epoch',

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=2,

    weight_decay=0.01,

    logging_dir='./logs',

    logging_steps=10
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

In [ ]:
import torch

print(torch.cuda.is_available())

print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.633100,0.686416,0.675833,0.677391,0.680114,0.675833
2,0.438750,0.724834,0.676667,0.679217,0.682546,0.676667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1200, training_loss=0.6554786845048268, metrics={'train_runtime': 527.3677, 'train_samples_per_second': 18.204, 'train_steps_per_second': 2.275, 'total_flos': 1271709705830400.0, 'train_loss': 0.6554786845048268, 'epoch': 2.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.724833607673645,
 'eval_accuracy': 0.6766666666666666,
 'eval_f1': 0.6792166151618854,
 'eval_precision': 0.6825463358213888,
 'eval_recall': 0.6766666666666666,
 'eval_runtime': 19.1677,
 'eval_samples_per_second': 62.605,
 'eval_steps_per_second': 7.826,
 'epoch': 2.0}

In [ ]:
model.save_pretrained('/content/drive/MyDrive/distilbert_sentiment_model')

tokenizer.save_pretrained('/content/drive/MyDrive/distilbert_sentiment_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/distilbert_sentiment_model/tokenizer_config.json',
 '/content/drive/MyDrive/distilbert_sentiment_model/tokenizer.json')